# 01a0 — Blockchain & smart contracts primer

A ground-up, hands-on tour. We drive a real local Ethereum chain (`anvil`) through `cast` and `forge`, naming each concept after we've executed it. By the last section you'll be able to read [BandwidthEscrow.sol](../../contracts/src/BandwidthEscrow.sol) line by line.

**Prereq:** `anvil`, `cast`, `forge` on PATH (Foundry installed). Run cells top to bottom — anvil is started in the next cell and killed in the very last cell.

In [1]:
# --- Notebook runtime setup ---------------------------------------
import atexit, subprocess, time, shutil, sys, pathlib, json, os

PRIMER_DIR = pathlib.Path.cwd().resolve()
REPO_ROOT = PRIMER_DIR.parent.parent
RPC = 'http://127.0.0.1:8545'

def run(cmd, cwd=None, check=True):
    """Run a shell command, show it, return stdout."""
    print('$', ' '.join(str(c) for c in cmd))
    r = subprocess.run(cmd, cwd=cwd or PRIMER_DIR, capture_output=True, text=True)
    if r.stdout: print(r.stdout.rstrip())
    if r.returncode != 0:
        if r.stderr: print(r.stderr.rstrip(), file=sys.stderr)
        if check: raise SystemExit(f'command failed: {cmd}')
    return r.stdout.strip()

for tool in ('anvil', 'cast', 'forge'):
    assert shutil.which(tool), f'{tool} not found on PATH'
print('Foundry tools OK')

Foundry tools OK


In [2]:
# --- Start anvil --------------------------------------------------
_anvil_proc = subprocess.Popen(
    ['anvil', '--host', '127.0.0.1', '--port', '8545', '--silent'],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
atexit.register(_anvil_proc.terminate)

# Wait for RPC to respond.
for _ in range(30):
    try:
        run(['cast', 'block-number', '--rpc-url', RPC], check=True)
        break
    except SystemExit:
        time.sleep(0.2)
else:
    raise RuntimeError('anvil did not come up')
print(f'anvil PID={_anvil_proc.pid}')

$ cast block-number --rpc-url http://127.0.0.1:8545


Error: error sending request for url (http://127.0.0.1:8545/)

Context:
- Error #0: client error (Connect)
- Error #1: tcp connect error
- Error #2: Connection refused (os error 111)


$ cast block-number --rpc-url http://127.0.0.1:8545
0
anvil PID=143574


## 1. What is a chain, really

You already know the intuition: a blockchain is an append-only distributed database of transactions. Let's make that concrete.

We started `anvil` — a process that pretends to be the entire Ethereum network. One node, no peers, no proof-of-stake. It exposes the same JSON-RPC interface mainnet does, on `http://127.0.0.1:8545`.

The chain has two things we'll keep separate in our heads:

1. **State** — current balances and contract storage (the "database").
2. **History** — the ordered list of blocks, each containing the    transactions that produced the next state.

Let's poke at both.

In [3]:
block_number = run(['cast', 'block-number', '--rpc-url', RPC])
print(f'\ncurrent block number: {block_number}')

$ cast block-number --rpc-url http://127.0.0.1:8545
0

current block number: 0


In [4]:
# The block itself. Block 0 is the genesis block — empty, no parent.
run(['cast', 'block', '0', '--rpc-url', RPC])

$ cast block 0 --rpc-url http://127.0.0.1:8545


baseFeePerGas        1000000000
difficulty           0
extraData            0x
gasLimit             30000000
gasUsed              0
hash                 0x672692712bd9c9401b2cb26950cbcc84ee997635b2ea51b7381192f0d07d8cff
logsBloom            0x00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000
miner                0x0000000000000000000000000000000000000000
mixHash              0x0000000000000000000000000000000000000000000000000000000000000000
nonce                0x0000000000000000
num

'baseFeePerGas        1000000000\ndifficulty           0\nextraData            0x\ngasLimit             30000000\ngasUsed              0\nhash                 0x672692712bd9c9401b2cb26950cbcc84ee997635b2ea51b7381192f0d07d8cff\nlogsBloom            0x00000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000\nminer                0x0000000000000000000000000000000000000000\nmixHash              0x0000000000000000000000000000000000000000000000000000000000000000\nnonce                0x0000000000000000\nnumber               0\nparentHash       

Notice the fields: `number`, `timestamp`, `parentHash`, `stateRoot`, `transactionsRoot`. Each block points to its parent by hash — that's the "chain" part. The `stateRoot` is a Merkle root summarising the entire state at this block — change one balance, the root changes, and so does the block hash.

## Teardown

Kill the anvil process. Re-run this notebook from the top to start fresh.

In [5]:
_anvil_proc.terminate()
_anvil_proc.wait(timeout=5)
print('anvil stopped')

anvil stopped
